<a href="https://colab.research.google.com/github/vitor-michelucci/postech-tech-challenge-fase2-template/blob/main/notebooks/00_preparacao_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 — Preparação dos Dados

Este notebook realiza a preparação inicial dos dados utilizados no Tech Challenge — Fase 2.

## Objetivos

- Obter os dados originais a partir da fonte pública.
- Verificar a estrutura das bases disponíveis.
- Analisar a relação entre os identificadores das bases.
- Estudar o histórico de crédito necessário para definição da variável-alvo.
- Construir e justificar a variável-alvo.
- Consolidar as informações em uma base analítica para as etapas seguintes.

> **Fonte dos dados:** Credit Card Approval Prediction, disponibilizado no Kaggle.

In [1]:
# ============================================================
# 1. CONFIGURAÇÃO DO AMBIENTE
# ============================================================

from pathlib import Path

# Endereço do repositório do projeto
REPO_URL = "https://github.com/vitor-michelucci/postech-tech-challenge-fase2-template.git"

# Local onde o repositório ficará disponível no Colab
PROJECT_PATH = Path("/content/postech-tech-challenge-fase2-template")

# Clona o repositório somente se ele ainda não estiver disponível
if not PROJECT_PATH.exists():
    !git clone {REPO_URL}
else:
    print("Repositório já disponível nesta sessão.")

print("\nDiretório do projeto:")
print(PROJECT_PATH)

Repositório já disponível nesta sessão.

Diretório do projeto:
/content/postech-tech-challenge-fase2-template


In [2]:
# ============================================================
# 2. OBTENÇÃO DOS DADOS ORIGINAIS
# ============================================================

import shutil
import kagglehub

# Baixa o dataset ou utiliza a versão já disponível no cache
dataset_path = kagglehub.dataset_download(
    "rikdifos/credit-card-approval-prediction"
)

KAGGLE_PATH = Path(dataset_path)

# Pasta destinada aos dados originais dentro do projeto
RAW_PATH = PROJECT_PATH / "data" / "raw"

# Garante que a pasta exista
RAW_PATH.mkdir(parents=True, exist_ok=True)

# Arquivos originais necessários para o projeto
arquivos = [
    "application_record.csv",
    "credit_record.csv"
]

# Copia os arquivos para data/raw
for nome_arquivo in arquivos:
    origem = KAGGLE_PATH / nome_arquivo
    destino = RAW_PATH / nome_arquivo

    if not destino.exists():
        shutil.copy2(origem, destino)
        print(f"Copiado: {nome_arquivo}")
    else:
        print(f"Já disponível: {nome_arquivo}")

print("\nDados originais disponíveis em:")
print(RAW_PATH)

Using Colab cache for faster access to the 'credit-card-approval-prediction' dataset.
Já disponível: application_record.csv
Já disponível: credit_record.csv

Dados originais disponíveis em:
/content/postech-tech-challenge-fase2-template/data/raw


In [3]:
# ============================================================
# 3. VERIFICAÇÃO DOS ARQUIVOS ORIGINAIS
# ============================================================

for nome_arquivo in arquivos:
    arquivo = RAW_PATH / nome_arquivo

    tamanho_mb = arquivo.stat().st_size / (1024 ** 2)

    print(f"{nome_arquivo}: {tamanho_mb:.2f} MB")

application_record.csv: 51.83 MB
credit_record.csv: 14.66 MB


## 2 — Conhecimento inicial das bases

Nesta etapa, as duas bases originais são carregadas separadamente para análise de sua estrutura, dimensões, variáveis e tipos de dados.

Nenhuma transformação é realizada neste momento.

In [4]:
# ============================================================
# 4. CARREGAMENTO DAS BASES
# ============================================================

import pandas as pd

# Carrega os arquivos originais
df_application = pd.read_csv(
    RAW_PATH / "application_record.csv"
)

df_credit = pd.read_csv(
    RAW_PATH / "credit_record.csv"
)

print("Bases carregadas com sucesso.")

Bases carregadas com sucesso.


In [5]:
# Dimensões das bases
print("application_record:", df_application.shape)
print("credit_record:", df_credit.shape)

application_record: (438557, 18)
credit_record: (1048575, 3)


In [6]:
# ============================================================
# 5. COLUNAS DISPONÍVEIS EM CADA BASE
# ============================================================

print("Colunas de application_record:")
print(df_application.columns.tolist())

print("\nColunas de credit_record:")
print(df_credit.columns.tolist())

Colunas de application_record:
['ID', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS']

Colunas de credit_record:
['ID', 'MONTHS_BALANCE', 'STATUS']


In [7]:
# ============================================================
# 6. ANÁLISE DOS IDENTIFICADORES
# ============================================================

print("application_record")
print("Linhas:", len(df_application))
print("IDs únicos:", df_application["ID"].nunique())

print("\ncredit_record")
print("Linhas:", len(df_credit))
print("IDs únicos:", df_credit["ID"].nunique())

application_record
Linhas: 438557
IDs únicos: 438510

credit_record
Linhas: 1048575
IDs únicos: 45985


In [8]:
# ============================================================
# 7. RELAÇÃO ENTRE OS IDs DAS DUAS BASES
# ============================================================

# Conjuntos de IDs únicos de cada base
ids_application = set(df_application["ID"])
ids_credit = set(df_credit["ID"])

# IDs presentes nas duas bases
ids_comuns = ids_application.intersection(ids_credit)

print("IDs únicos em application_record:", len(ids_application))
print("IDs únicos em credit_record:", len(ids_credit))
print("IDs presentes nas duas bases:", len(ids_comuns))

IDs únicos em application_record: 438510
IDs únicos em credit_record: 45985
IDs presentes nas duas bases: 36457


In [9]:
# ============================================================
# 8. INVESTIGAÇÃO DE IDs DUPLICADOS EM application_record
# ============================================================

# Identifica todas as linhas cujo ID aparece mais de uma vez
duplicados_application = df_application[
    df_application["ID"].duplicated(keep=False)
].sort_values("ID")

print("Quantidade de linhas com IDs duplicados:",
      len(duplicados_application))

print("Quantidade de IDs envolvidos:",
      duplicados_application["ID"].nunique())

Quantidade de linhas com IDs duplicados: 94
Quantidade de IDs envolvidos: 47


In [10]:
# ============================================================
# 9. VERIFICAÇÃO DOS REGISTROS COM IDs DUPLICADOS
# ============================================================

# Exibe os registros dos primeiros IDs duplicados
duplicados_application.head(10)

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
425023,7022197,F,N,Y,0,450000.0,Commercial associate,Higher education,Separated,House / apartment,-19813,-1799,1,0,0,1,NaN,1.0
426818,7022197,M,Y,Y,3,135000.0,Working,Secondary / secondary special,Married,House / apartment,-11945,-735,1,0,0,1,Laborers,5.0
431911,7022327,M,Y,Y,0,256500.0,Commercial associate,Higher education,Married,House / apartment,-21503,-1674,1,0,0,1,Core staff,2.0
431545,7022327,F,N,Y,0,135000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-14771,-5298,1,0,0,0,High skill tech staff,1.0
425486,7023108,M,Y,Y,1,67500.0,Working,Secondary / secondary special,Married,House / apartment,-15156,-1696,1,1,0,0,Core staff,3.0
426488,7023108,F,N,N,0,135000.0,Working,Secondary / secondary special,Married,House / apartment,-17590,-1273,1,0,0,0,Cleaning staff,2.0
425306,7023651,F,N,N,0,225000.0,Commercial associate,Incomplete higher,Single / not married,House / apartment,-10229,-1209,1,0,0,0,Accountants,1.0
421907,7023651,M,Y,N,1,157500.0,Commercial associate,Incomplete higher,Married,House / apartment,-10521,-1457,1,0,0,0,Drivers,3.0
427778,7024111,M,N,N,2,157500.0,Commercial associate,Secondary / secondary special,Civil marriage,House / apartment,-15270,-117,1,1,0,0,Drivers,4.0
432643,7024111,F,N,Y,0,180000.0,Working,Secondary / secondary special,Married,House / apartment,-22041,-1524,1,1,1,0,Accountants,2.0


In [11]:
# ============================================================
# 10. IDs DUPLICADOS QUE POSSUEM HISTÓRICO DE CRÉDITO
# ============================================================

# IDs que aparecem mais de uma vez na base cadastral
ids_duplicados = set(duplicados_application["ID"])

# Verifica quais deles também aparecem na base de crédito
ids_duplicados_com_credito = ids_duplicados.intersection(ids_credit)

print("IDs duplicados em application_record:",
      len(ids_duplicados))

print("IDs duplicados que também aparecem em credit_record:",
      len(ids_duplicados_com_credito))

IDs duplicados em application_record: 47
IDs duplicados que também aparecem em credit_record: 0


### Análise dos identificadores

A base `application_record` possui 438.557 registros e 438.510 IDs únicos. Foram identificados 47 IDs repetidos, cada um associado a dois registros cadastrais distintos.

A inspeção desses casos mostrou que não se tratam de duplicatas exatas, pois existem diferenças entre as características cadastrais associadas ao mesmo identificador.

Entretanto, nenhum dos 47 IDs duplicados está presente na base `credit_record`. Dessa forma, esses registros não pertencem ao conjunto de clientes para os quais existe histórico de crédito e, consequentemente, não integrarão a base analítica utilizada para construção da variável-alvo e posterior modelagem.

As bases possuem 36.457 IDs em comum.

## 3 — Análise do histórico de crédito

A base `credit_record` contém o histórico mensal de pagamento dos clientes.

Nesta etapa, serão analisadas as variáveis `MONTHS_BALANCE` e `STATUS`, com foco no período utilizado para a construção da variável-alvo.

Para garantir comparabilidade entre os clientes, a população de modelagem será composta somente por clientes que possuem **seis meses consecutivos de histórico**, correspondentes aos meses `0, -1, -2, -3, -4 e -5`.

A variável-alvo será construída a partir do comportamento observado nessa janela de seis meses.

In [12]:
# ============================================================
# 11. PRIMEIROS REGISTROS DO HISTÓRICO DE CRÉDITO
# ============================================================

df_credit.head(10)

,ID,MONTHS_BALANCE,STATUS
0,5001711,0,X
1,5001711,-1,0
2,5001711,-2,0
3,5001711,-3,0
4,5001712,0,C
5,5001712,-1,C
6,5001712,-2,C
7,5001712,-3,C
8,5001712,-4,C
9,5001712,-5,C


In [13]:
# ============================================================
# 12. DISTRIBUIÇÃO DA VARIÁVEL STATUS
# ============================================================

# Contagem de registros para cada situação de crédito
status_counts = (
    df_credit["STATUS"]
    .value_counts()
    .sort_index()
)

print(status_counts)

STATUS
0    383120
1     11090
2       868
3       320
4       223
5      1693
C    442031
X    209230
Name: count, dtype: int64


In [14]:
# Percentual de cada categoria de STATUS
status_percentual = (
    df_credit["STATUS"]
    .value_counts(normalize=True)
    .mul(100)
    .sort_index()
    .round(2)
)

print(status_percentual)

STATUS
0    36.54
1     1.06
2     0.08
3     0.03
4     0.02
5     0.16
C    42.16
X    19.95
Name: proportion, dtype: float64


In [15]:
# ============================================================
# 13. QUANTIDADE DE MESES OBSERVADOS POR CLIENTE
# ============================================================

meses_por_cliente = (
    df_credit
    .groupby("ID")
    .size()
)

print(meses_por_cliente.describe())

count    45985.000000
mean        22.802544
std         15.492771
min          1.000000
25%         10.000000
50%         19.000000
75%         34.000000
max         61.000000
dtype: float64


In [16]:
# ============================================================
# 14. INTERVALO TEMPORAL DO HISTÓRICO
# ============================================================

print("Maior valor de MONTHS_BALANCE:",
      df_credit["MONTHS_BALANCE"].max())

print("Menor valor de MONTHS_BALANCE:",
      df_credit["MONTHS_BALANCE"].min())

print("Quantidade de valores distintos:",
      df_credit["MONTHS_BALANCE"].nunique())

Maior valor de MONTHS_BALANCE: 0
Menor valor de MONTHS_BALANCE: -60
Quantidade de valores distintos: 61


## 4 — Definição da variável-alvo

A variável-alvo foi construída a partir do histórico de crédito observado em uma janela fixa de seis meses, compreendendo os meses `0, -1, -2, -3, -4 e -5`.

O cliente será classificado como **evento (`target = 1`)** quando atender a pelo menos uma das seguintes condições:

- apresentar atraso igual ou superior a 30 dias no mês `0` (`STATUS >= 1`); **ou**
- apresentar atraso igual ou superior a 90 dias em qualquer um dos seis meses da janela `0:-5` (`STATUS >= 3`); **ou**
- apresentar atraso igual ou superior a 30 dias em pelo menos dois meses da janela `0:-5` (`STATUS >= 1` em pelo menos dois meses).

O cliente será classificado como **não evento (`target = 0`)** quando não atender a nenhuma dessas condições.

Para a construção da variável-alvo, os registros `C` e `X` não são considerados eventos de atraso. O `X` representa ausência de empréstimo registrado naquele mês e, portanto, não constitui evidência de inadimplência.

Somente clientes que possuem histórico completo nos seis meses da janela `0:-5` e que também estão presentes na base cadastral serão considerados na população de modelagem. Dessa forma, a regra é aplicada sobre uma janela temporal padronizada para todos os clientes.

In [17]:
# ============================================================
# 15. DEFINIÇÃO DA JANELA DE OBSERVAÇÃO
# ============================================================

# A construção da target utiliza seis meses consecutivos:
# mês atual (0) e os cinco meses anteriores (-1 a -5).

MESES_TARGET = [0, -1, -2, -3, -4, -5]

print("Meses utilizados na construção da target:")
print(MESES_TARGET)

print(f"\nQuantidade de meses da janela: {len(MESES_TARGET)}")

Meses utilizados na construção da target:
[0, -1, -2, -3, -4, -5]

Quantidade de meses da janela: 6


In [18]:
# ============================================================
# 16. SELEÇÃO DA JANELA DE HISTÓRICO
# ============================================================

# Mantém somente os registros pertencentes
# aos seis meses utilizados na construção da target.

df_credit_6m = df_credit[
    df_credit["MONTHS_BALANCE"].isin(MESES_TARGET)
].copy()

print("Dimensões do histórico na janela:")
print(df_credit_6m.shape)

print(
    f"\nClientes distintos na janela: "
    f"{df_credit_6m['ID'].nunique():,}"
)

Dimensões do histórico na janela:
(202601, 3)

Clientes distintos na janela: 36,501


In [19]:
# ============================================================
# 17. IDENTIFICAÇÃO DOS CLIENTES COM HISTÓRICO COMPLETO
# ============================================================

# Conta quantos meses distintos cada cliente possui
# dentro da janela de 0 a -5.

meses_por_cliente = (
    df_credit_6m
    .groupby("ID")["MONTHS_BALANCE"]
    .nunique()
)

# Mantém somente clientes que possuem os seis meses
# da janela de forma completa.

ids_historico_completo = meses_por_cliente[
    meses_por_cliente == len(MESES_TARGET)
].index

print(
    f"Clientes com histórico completo de 0 a -5: "
    f"{len(ids_historico_completo):,}"
)

Clientes com histórico completo de 0 a -5: 30,340


In [20]:
# ============================================================
# 18. HISTÓRICO DA POPULAÇÃO COM SEIS MESES COMPLETOS
# ============================================================

df_credit_6m = df_credit_6m[
    df_credit_6m["ID"].isin(ids_historico_completo)
].copy()

print(
    f"Clientes mantidos: "
    f"{df_credit_6m['ID'].nunique():,}"
)

print(
    f"Registros mantidos: "
    f"{len(df_credit_6m):,}"
)

Clientes mantidos: 30,340
Registros mantidos: 182,040


## 5 — Definição da população de modelagem

A população de modelagem é formada pela interseção entre os clientes que possuem histórico de crédito completo nos seis meses da janela `0:-5` e os clientes presentes na base cadastral.

Clientes com histórico completo, mas sem informações cadastrais, não poderão ser utilizados na modelagem, pois não possuem as variáveis explicativas necessárias.

In [21]:
# ============================================================
# 19. INTERSEÇÃO ENTRE HISTÓRICO E CADASTRO
# ============================================================

# IDs presentes na base cadastral
ids_cadastro = set(df_application["ID"].unique())

# IDs que possuem histórico completo E cadastro
ids_modelagem = (
    set(ids_historico_completo)
    .intersection(ids_cadastro)
)

print("=" * 60)
print("POPULAÇÃO DE MODELAGEM")
print("=" * 60)

print(
    f"Clientes com histórico completo: "
    f"{len(ids_historico_completo):,}"
)

print(
    f"Clientes presentes no cadastro: "
    f"{len(ids_cadastro):,}"
)

print(
    f"Clientes com histórico completo e cadastro: "
    f"{len(ids_modelagem):,}"
)

print(
    f"Clientes com histórico completo, mas sem cadastro: "
    f"{len(ids_historico_completo) - len(ids_modelagem):,}"
)

POPULAÇÃO DE MODELAGEM
Clientes com histórico completo: 30,340
Clientes presentes no cadastro: 438,510
Clientes com histórico completo e cadastro: 21,694
Clientes com histórico completo, mas sem cadastro: 8,646


In [22]:
# ============================================================
# 20. HISTÓRICO DA POPULAÇÃO DE MODELAGEM
# ============================================================

df_credit_6m = df_credit_6m[
    df_credit_6m["ID"].isin(ids_modelagem)
].copy()

print(
    f"Clientes na população de modelagem: "
    f"{df_credit_6m['ID'].nunique():,}"
)

print(
    f"Registros de histórico mantidos: "
    f"{len(df_credit_6m):,}"
)

Clientes na população de modelagem: 21,694
Registros de histórico mantidos: 130,164


In [23]:
# ============================================================
# 21. VALIDAÇÃO DO HISTÓRICO DA POPULAÇÃO DE MODELAGEM
# ============================================================

meses_por_cliente_modelagem = (
    df_credit_6m
    .groupby("ID")["MONTHS_BALANCE"]
    .nunique()
)

print(
    "Quantidade de clientes com exatamente 6 meses:",
    (meses_por_cliente_modelagem == 6).sum()
)

print(
    "Quantidade de clientes com menos de 6 meses:",
    (meses_por_cliente_modelagem < 6).sum()
)

print(
    "Quantidade de clientes com mais de 6 meses:",
    (meses_por_cliente_modelagem > 6).sum()
)

Quantidade de clientes com exatamente 6 meses: 21694
Quantidade de clientes com menos de 6 meses: 0
Quantidade de clientes com mais de 6 meses: 0


In [24]:
# ============================================================
# 22. VERIFICAÇÃO DE DUPLICIDADES CLIENTE × MÊS
# ============================================================

duplicidades = (
    df_credit_6m
    .groupby(["ID", "MONTHS_BALANCE"])
    .size()
)

duplicidades = duplicidades[duplicidades > 1]

print(
    f"Combinações ID + mês duplicadas: "
    f"{len(duplicidades):,}"
)

if len(duplicidades) > 0:
    print("\nAtenção: existem duplicidades.")
    display(duplicidades.head(10))

Combinações ID + mês duplicadas: 0


In [25]:
# ============================================================
# 23. PREPARAÇÃO DA VARIÁVEL STATUS
# ============================================================

# STATUS possui valores categóricos (C, X e 0 a 5).
# Para identificar os níveis de atraso, convertemos apenas
# os valores numéricos.
#
# C = situação regular
# X = ausência de empréstimo registrado no mês
# 0 = atraso de 1 a 29 dias
# 1 = atraso de 30 a 59 dias
# 2 = atraso de 60 a 89 dias
# 3 = atraso de 90 a 119 dias
# 4 = atraso de 120 a 149 dias
# 5 = atraso de 150 dias ou mais
#
# C e X serão convertidos para NaN e não serão considerados
# eventos de atraso.

df_credit_6m["STATUS_NUM"] = pd.to_numeric(
    df_credit_6m["STATUS"],
    errors="coerce"
)

print("Distribuição do STATUS:")
print(df_credit_6m["STATUS"].value_counts().sort_index())

Distribuição do STATUS:
STATUS
0    36897
1     1353
2      118
3       42
4       40
5      297
C    70317
X    21100
Name: count, dtype: int64


In [26]:
# ============================================================
# 24. CRITÉRIO A — ATRASO >=30 DIAS NO MÊS 0
# ============================================================

# STATUS 1, 2, 3, 4 ou 5 representa atraso de 30 dias ou mais.

ids_criterio_a = set(
    df_credit_6m.loc[
        (df_credit_6m["MONTHS_BALANCE"] == 0) &
        (df_credit_6m["STATUS_NUM"] >= 1),
        "ID"
    ]
)

print(
    f"Clientes com atraso >=30 dias no mês 0: "
    f"{len(ids_criterio_a):,}"
)

Clientes com atraso >=30 dias no mês 0: 273


In [27]:
# ============================================================
# 25. CRITÉRIO B — ATRASO >=90 DIAS NA JANELA
# ============================================================

# STATUS 3, 4 ou 5 representa atraso de 90 dias ou mais.
#
# Basta ocorrer uma vez em qualquer dos seis meses para
# o cliente atender ao critério.

ids_criterio_b = set(
    df_credit_6m.loc[
        df_credit_6m["STATUS_NUM"] >= 3,
        "ID"
    ]
)

print(
    f"Clientes com atraso >=90 dias em 0:-5: "
    f"{len(ids_criterio_b):,}"
)

Clientes com atraso >=90 dias em 0:-5: 104


In [28]:
# ============================================================
# 26. CRITÉRIO C — ATRASO >=30 DIAS EM PELO MENOS 2 MESES
# ============================================================

# Conta, para cada cliente, em quantos dos seis meses
# ocorreu STATUS >= 1.

meses_com_atraso_30 = (
    df_credit_6m
    .assign(
        atraso_30=df_credit_6m["STATUS_NUM"] >= 1
    )
    .groupby("ID")["atraso_30"]
    .sum()
)

# Cliente atende ao critério quando apresenta atraso >=30
# em pelo menos dois meses da janela.

ids_criterio_c = set(
    meses_com_atraso_30[
        meses_com_atraso_30 >= 2
    ].index
)

print(
    f"Clientes com atraso >=30 em pelo menos 2 meses: "
    f"{len(ids_criterio_c):,}"
)

Clientes com atraso >=30 em pelo menos 2 meses: 420


In [29]:
# ============================================================
# 27. RESUMO DOS CRITÉRIOS DA TARGET
# ============================================================

print("=" * 60)
print("RESUMO DOS CRITÉRIOS")
print("=" * 60)

print(
    f"Critério A - >=30 dias no mês 0: "
    f"{len(ids_criterio_a):,}"
)

print(
    f"Critério B - >=90 dias em 0:-5: "
    f"{len(ids_criterio_b):,}"
)

print(
    f"Critério C - >=30 dias em pelo menos 2 meses: "
    f"{len(ids_criterio_c):,}"
)

RESUMO DOS CRITÉRIOS
Critério A - >=30 dias no mês 0: 273
Critério B - >=90 dias em 0:-5: 104
Critério C - >=30 dias em pelo menos 2 meses: 420


In [30]:
# ============================================================
# 28. SOBREPOSIÇÃO ENTRE OS CRITÉRIOS
# ============================================================

print("=" * 60)
print("SOBREPOSIÇÃO ENTRE OS CRITÉRIOS")
print("=" * 60)

print(
    f"Apenas A: "
    f"{len(ids_criterio_a - ids_criterio_b - ids_criterio_c):,}"
)

print(
    f"Apenas B: "
    f"{len(ids_criterio_b - ids_criterio_a - ids_criterio_c):,}"
)

print(
    f"Apenas C: "
    f"{len(ids_criterio_c - ids_criterio_a - ids_criterio_b):,}"
)

print(
    f"A + B: "
    f"{len((ids_criterio_a & ids_criterio_b) - ids_criterio_c):,}"
)

print(
    f"A + C: "
    f"{len((ids_criterio_a & ids_criterio_c) - ids_criterio_b):,}"
)

print(
    f"B + C: "
    f"{len((ids_criterio_b & ids_criterio_c) - ids_criterio_a):,}"
)

print(
    f"A + B + C: "
    f"{len(ids_criterio_a & ids_criterio_b & ids_criterio_c):,}"
)

SOBREPOSIÇÃO ENTRE OS CRITÉRIOS
Apenas A: 76
Apenas B: 13
Apenas C: 202
A + B: 0
A + C: 127
B + C: 21
A + B + C: 70


In [31]:
# ============================================================
# 29. CONSTRUÇÃO DA TARGET DEFINITIVA
# ============================================================

# A Alternativa 2 utiliza a união dos três critérios:
#
# A = atraso >=30 dias no mês 0
# B = atraso >=90 dias em qualquer mês
# C = atraso >=30 dias em pelo menos 2 meses
#
# Se qualquer condição for satisfeita, target = 1.

ids_target_1 = (
    ids_criterio_a
    | ids_criterio_b
    | ids_criterio_c
)

print(
    f"Total de clientes classificados como evento: "
    f"{len(ids_target_1):,}"
)

Total de clientes classificados como evento: 509


In [32]:
# ============================================================
# 30. DATAFRAME FINAL DA TARGET
# ============================================================

df_target = pd.DataFrame({
    "ID": list(ids_modelagem)
})

df_target["target"] = (
    df_target["ID"]
    .isin(ids_target_1)
    .astype(int)
)

print("=" * 60)
print("DISTRIBUIÇÃO FINAL DA TARGET")
print("=" * 60)

print(df_target["target"].value_counts())

print("\nPercentual:")

print(
    df_target["target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

DISTRIBUIÇÃO FINAL DA TARGET
target
0    21185
1      509
Name: count, dtype: int64

Percentual:
target
0    97.65
1     2.35
Name: proportion, dtype: float64


## 6 — Consolidação da base de modelagem

Após a definição da população e da variável-alvo a partir do histórico de crédito, a target será integrada às informações cadastrais dos clientes.

A base cadastral será utilizada como fonte das variáveis explicativas que serão exploradas na EDA e posteriormente utilizadas na modelagem.

In [33]:
# ============================================================
# 31. PREPARAÇÃO DA BASE CADASTRAL
# ============================================================

# Cada cliente deve possuir apenas um registro cadastral
# para que o merge com a target seja do tipo 1:1.

df_application_unico = (
    df_application
    .drop_duplicates(subset="ID")
    .copy()
)

print("Registros antes da remoção de duplicidades:",
      len(df_application))

print("Registros após a remoção de duplicidades:",
      len(df_application_unico))

print("IDs únicos:",
      df_application_unico["ID"].nunique())

Registros antes da remoção de duplicidades: 438557
Registros após a remoção de duplicidades: 438510
IDs únicos: 438510


In [34]:
# ============================================================
# 32. MERGE DA BASE CADASTRAL COM A TARGET
# ============================================================

df_modelagem = (
    df_application_unico[
        df_application_unico["ID"].isin(ids_modelagem)
    ]
    .merge(
        df_target,
        on="ID",
        how="inner",
        validate="one_to_one"
    )
)

print("=" * 60)
print("BASE FINAL DE MODELAGEM")
print("=" * 60)

print(f"Dimensões: {df_modelagem.shape}")

print(
    f"Clientes distintos: "
    f"{df_modelagem['ID'].nunique():,}"
)

print("\nDistribuição da target:")
print(df_modelagem["target"].value_counts())

print("\nPercentual da target:")
print(
    df_modelagem["target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

BASE FINAL DE MODELAGEM
Dimensões: (21694, 19)
Clientes distintos: 21,694

Distribuição da target:
target
0    21185
1      509
Name: count, dtype: int64

Percentual da target:
target
0    97.65
1     2.35
Name: proportion, dtype: float64


In [35]:
# ============================================================
# 33. VALIDAÇÃO DA BASE FINAL
# ============================================================

print("=" * 60)
print("VALIDAÇÕES FINAIS")
print("=" * 60)

print(
    "IDs duplicados:",
    df_modelagem["ID"].duplicated().sum()
)

print(
    "Valores ausentes na target:",
    df_modelagem["target"].isna().sum()
)

print(
    "Clientes da população esperada:",
    len(ids_modelagem)
)

print(
    "Clientes na base final:",
    df_modelagem["ID"].nunique()
)

VALIDAÇÕES FINAIS
IDs duplicados: 0
Valores ausentes na target: 0
Clientes da população esperada: 21694
Clientes na base final: 21694


## 7 — Exportação da base analítica

A base final de modelagem, composta pelas informações cadastrais e pela variável-alvo construída a partir do histórico de crédito, é exportada para utilização nas etapas seguintes do projeto.

In [36]:
# ============================================================
# 34. EXPORTAÇÃO DA BASE ANALÍTICA
# ============================================================

# Pasta destinada às bases processadas
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"

# Garante que a pasta exista
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# Caminho da base que será utilizada pelo Notebook 01
DATASET_BASE_PATH = PROCESSED_PATH / "dataset_base.csv"

# Exporta a base final de modelagem
df_modelagem.to_csv(
    DATASET_BASE_PATH,
    index=False
)

print("=" * 60)
print("EXPORTAÇÃO DA BASE ANALÍTICA")
print("=" * 60)

print(f"Arquivo: {DATASET_BASE_PATH}")
print(f"Dimensões: {df_modelagem.shape}")
print(f"Clientes: {df_modelagem['ID'].nunique():,}")

EXPORTAÇÃO DA BASE ANALÍTICA
Arquivo: /content/postech-tech-challenge-fase2-template/data/processed/dataset_base.csv
Dimensões: (21694, 19)
Clientes: 21,694
